In [69]:
import numpy as np
import  pandas as pd 
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd
from geopy.distance import geodesic
import contextily as ctx
from sklearn.preprocessing import MinMaxScaler



In [70]:
import os
print(os.getcwd())

c:\Users\jorge_j24fcle\OneDrive\Documentos\3. coding\Python\GSNE-replication


In [71]:
houses =pd.read_csv("minimal_example/synthetic_data/houses_v2.csv")
regions =pd.read_csv("minimal_example/synthetic_data/regions.csv")
schools =pd.read_csv("minimal_example/synthetic_data/schools_v2.csv")
stations =pd.read_csv("minimal_example/synthetic_data/stations_v2.csv")


In [72]:
stations.head()

,poi_id,elevated,platform_length_m,station_zone,has_bike_parking,lat,lon,daily_traffic,connections
0,T1,yes,120,A,yes,4.680,-74.08,5000,4
1,T2,yes,115,A,yes,4.655,-74.10,4200,3
2,T3,no,130,B,no,4.700,-74.06,6100,5
3,T4,yes,110,B,yes,4.640,-74.12,3800,2
4,T5,no,140,C,no,4.720,-74.05,7300,6


In [73]:
G = nx.Graph()

In [74]:
for _, row in houses.iterrows():
    G.add_node(
     
     row['property_id'],
     type = 'house',
     price = row['price'],
     rooms = row['rooms'],
     bathrooms = row['bathrooms'],
     surface_total = row['surface_total'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [75]:
for _, row in stations.iterrows():
    G.add_node(
     
     row['poi_id'],
     type = 'station',
     daily_traffic = row['daily_traffic'],
     connections = row['connections'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [76]:
for _, row in schools.iterrows():
    G.add_node(
     row['poi_id'],
     type = 'school',
     ranking = row['ranking'],
     students = row['students'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [77]:
def add_edges(G, df1, df2):
    df1_id = df1.columns[df1.columns.str.endswith("_id")][0]
    df2_id = df2.columns[df2.columns.str.endswith("_id")][0]
    
    for _, row in df1.iterrows():
        coords_1 = (row['lat'], row['lon'])
        
        for _, rowa in df2.iterrows():
            coords_2 = (rowa['lat'], rowa['lon'])
            
            distance = geodesic(coords_1, coords_2).km
            
            if distance < 5:  # Connect if within 5 km
                
                G.add_edge(
                    row[df1_id], rowa[df2_id], weight= 1/distance)
    return G
            

In [78]:
G = add_edges(G, houses, stations)
G = add_edges(G, houses, schools)
G = add_edges(G, schools, stations)

In [79]:
G.edges()

EdgeView([('H0', 'T2'), ('H0', 'T4'), ('H0', 'S4'), ('H1', 'T2'), ('H1', 'T4'), ('H1', 'S4'), ('H3', 'T3'), ('H3', 'S3'), ('H4', 'T3'), ('H4', 'T5'), ('H4', 'S1'), ('H4', 'S3'), ('H5', 'T2'), ('H5', 'T4'), ('H5', 'S2'), ('H5', 'S4'), ('H6', 'T2'), ('H6', 'T4'), ('H6', 'S2'), ('H6', 'S4'), ('H7', 'T1'), ('H7', 'T3'), ('H7', 'T5'), ('H7', 'S1'), ('H7', 'S2'), ('H7', 'S3'), ('H8', 'T4'), ('T1', 'S1'), ('T1', 'S2'), ('T1', 'S3'), ('T2', 'S1'), ('T2', 'S2'), ('T2', 'S4'), ('T3', 'S1'), ('T3', 'S3'), ('T4', 'S2'), ('T4', 'S4'), ('T5', 'S3')])

In [80]:
def build_X_features(df):
    df1 = df.copy()

    # 1. Drop ID + lat/lon
    id_cols = [c for c in df1.columns if c.endswith("_id")]
    df1 = df1.drop(columns=id_cols + ["lat", "lon"], errors="ignore")

    # 2. Separate numeric + categorical
    numeric_cols = df1.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df1.select_dtypes(include=["object", "category", "bool"]).columns

    # 3. Impute missing values
    df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
    df1[cat_cols] = df1[cat_cols].fillna("Unknown")

    # 4. One-hot encode categoricals
    df1 = pd.get_dummies(df1, columns=cat_cols, drop_first=False)

    # 5. Scale numeric columns (per type!)
    scaler = MinMaxScaler()
    df1[numeric_cols] = scaler.fit_transform(df1[numeric_cols])

    return df1
